In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
EMB_DIR = ROOT / "embeddings-siglip2" / "embeddings-siglip2"
THUMB_DIR_RE = ROOT / "images-recatalogued-preview" / "images-recatalogued-preview"
THUM_DIR_LEG = ROOT / "images-legacy-preview" / "images-legacy-preview"
CACHE_DIR = ROOT / "cache"
CACHE_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

In [2]:
emb = np.load(EMB_DIR / "embeddings.npy")
files = pd.read_csv(EMB_DIR / "dateien.csv", encoding="utf-8-sig")
content = pd.read_csv(ROOT / "photochrom-image-content.csv", encoding="utf-8-sig")

assert len(files) == len(emb), "embedding rows and dateien.csv rows must match"

# Join content info (genre, English description) on filename + subset
content_cols = content[["image_filename", "subset", "genre", "beschreibung_en"]].drop_duplicates(
    subset=["image_filename", "subset"]
)
df = files.merge(content_cols, on=["image_filename", "subset"], how="left")

# Thumbnails are available for the recatalogued subset only
df["thumb_path"] = [
    str(THUMB_DIR_RE / fn) if subset == "recatalogued" and (THUMB_DIR_RE / fn).exists() else str(THUM_DIR_LEG / fn) if subset == "legacy" and (THUM_DIR_LEG / fn).exists() else None
    for fn, subset in zip(df["image_filename"], df["subset"])
]

# L2-normalize embeddings -> cosine similarity == dot product
X = emb / np.linalg.norm(emb, axis=1, keepdims=True)

print(f"embeddings: {emb.shape}, dtype={emb.dtype}")
print(f"rows with content info: {df['beschreibung_en'].notna().sum()}")
print(f"rows with thumbnail:    {df['thumb_path'].notna().sum()}")
df.head(3)

embeddings: (9582, 1152), dtype=float32
rows with content info: 9582
rows with thumbnail:    9582


,zeile,image_filename,subset,genre,beschreibung_en,thumb_path
0,0,000006639_14.jpg,legacy,landschaft,A photochrom of a mountain lake with a large w...,/home/kien/photochrom/images-legacy-preview/im...
1,1,000006639_15.jpg,legacy,landschaft,A photochrom of a winding road built into a st...,/home/kien/photochrom/images-legacy-preview/im...
2,2,000006639_16.jpg,legacy,landschaft,A photochrom photograph of a narrow gorge or c...,/home/kien/photochrom/images-legacy-preview/im...


In [3]:
import umap

def cached_umap(X, n_components, min_dist, cache_file):
    if cache_file.exists():
        print(f"loading cached {cache_file.name}")
        return np.load(cache_file)
    reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=30,
        min_dist=min_dist,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    out = reducer.fit_transform(X)
    np.save(cache_file, out)
    return out

X_cluster = cached_umap(X, 10, 0.0, CACHE_DIR / "umap_10d.npy")   # for HDBSCAN
X_2d      = cached_umap(X, 2, 0.1, CACHE_DIR / "umap_2d.npy")     # for plotting
print("10-D:", X_cluster.shape, " 2-D:", X_2d.shape)

/home/kien/photochrom/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading cached umap_10d.npy
loading cached umap_2d.npy
10-D: (9582, 10)  2-D: (9582, 2)


In [4]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=40,
    min_samples=10,
    cluster_selection_method="eom",
)
df["cluster"] = clusterer.fit_predict(X_cluster)

n_clusters = df["cluster"].max() + 1
n_noise = (df["cluster"] == -1).sum()
print(f"{n_clusters} clusters, {n_noise} noise points ({n_noise / len(df):.1%})")

sizes = df[df["cluster"] >= 0]["cluster"].value_counts().sort_index()
sizes.to_frame("size").T

44 clusters, 1393 noise points (14.5%)


cluster,0,1,2,3,4,5,6,7,8,9,...,34,35,36,37,38,39,40,41,42,43
size,114,43,2639,62,49,65,40,111,57,93,...,72,52,102,211,72,157,137,47,59,232


In [8]:
csv_labels_path = Path("photochrom/data/cluster-analysis/clusters_with_labels.csv")

label_df = pd.read_csv(csv_labels_path)

In [9]:
import base64
from io import BytesIO

from PIL import Image

THUMB_SIZE = 140  # max edge in px for the tooltip thumbnails

def thumb_data_uri(path):
    """Small base64 JPEG for embedding in the hover tooltip."""
    with Image.open(path) as im:
        im.thumbnail((THUMB_SIZE, THUMB_SIZE))
        buf = BytesIO()
        im.convert("RGB").save(buf, format="JPEG", quality=70)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

# Pre-render an <img> tag per point (empty string where no thumbnail exists)
thumb_html = []
for p in df["thumb_path"]:
    if pd.isna(p):
        thumb_html.append("")
    else:
        thumb_html.append(f'<img src="{thumb_data_uri(p)}" style="margin:4px 0;">')
df["thumb_html"] = thumb_html
print(f"embedded {sum(bool(h) for h in thumb_html)} thumbnails")

embedded 9582 thumbnails


In [12]:
from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource, HoverTool, LabelSet
from bokeh.palettes import Turbo256
from bokeh.plotting import figure
from scipy.spatial import cKDTree

output_notebook()

# color per cluster, light grey for noise
labels = df["cluster"].to_numpy()
palette = [Turbo256[int(i)] for i in np.linspace(5, 250, max(n_clusters, 1))]
# shuffle so that neighboring clusters (often consecutive IDs) get distinct colors
palette = [str(c) for c in np.random.default_rng(RANDOM_STATE).permutation(palette)]
colors = np.where(labels >= 0, [palette[l % len(palette)] for l in np.clip(labels, 0, None)], "#cccccc")

# --- light background area per cluster --------------------------------------
# Every grid cell gets the light color of the nearest clustered point, but only
# if that point is within BG_RADIUS -> cluster-shaped, non-overlapping areas.
GRID = 500
BG_RADIUS_FRAC = 0.025   # radius as fraction of the plot extent
LIGHTNESS = 0.78         # 0 = full cluster color, 1 = white

def hex_to_rgb(h):
    return np.array([int(h[i:i + 2], 16) for i in (1, 3, 5)])

clustered = labels >= 0
pts, pt_labels = X_2d[clustered], labels[clustered]

pad = 0.05 * np.ptp(X_2d, axis=0)
x0, y0 = X_2d.min(axis=0) - pad
x1, y1 = X_2d.max(axis=0) + pad
gx, gy = np.meshgrid(np.linspace(x0, x1, GRID), np.linspace(y0, y1, GRID))

dist, idx = cKDTree(pts).query(np.c_[gx.ravel(), gy.ravel()])
grid_labels = np.where(dist <= BG_RADIUS_FRAC * max(x1 - x0, y1 - y0), pt_labels[idx], -1)

light = np.array([(1 - LIGHTNESS) * hex_to_rgb(c) + LIGHTNESS * 255 for c in palette], dtype=np.uint8)
rgba = np.zeros((GRID * GRID, 4), dtype=np.uint8)   # transparent where no cluster
inside = grid_labels >= 0
rgba[inside, :3] = light[grid_labels[inside] % len(palette)]
rgba[inside, 3] = 255
bg_image = rgba.view(np.uint32).reshape(GRID, GRID)

# --- one genre label per cluster ----------------------------------------------
# Place the label on the cluster point closest to the cluster's median, so it
# always lies inside the cluster even for non-convex shapes.
#label_rows = []
#for c in range(n_clusters):
#    members = X_2d[labels == c]
#    center = members[np.argmin(np.linalg.norm(members - np.median(members, axis=0), axis=1))]
#    genres = df.loc[labels == c, "genre"].dropna()
#    label_rows.append((center[0], center[1], genres.mode().iat[0] if len(genres) else f"cluster {c}"))
#label_source = ColumnDataSource(dict(zip(("x", "y", "text"), map(list, zip(*label_rows)))))


# --- one label per cluster ----------------------------------------------
# Place the label on the cluster point closest to the cluster's median, so it
# always lies inside the cluster even for non-convex shapes.

label_rows = []
for c in range(n_clusters):
    members = X_2d[labels == c]
    center = members[np.argmin(np.linalg.norm(members - np.median(members, axis=0), axis=1))]

    cluster_labels = label_df.loc[labels == c, "label"].dropna()
    label_rows.append(
        (center[0], center[1], cluster_labels.iloc[0] if len(cluster_labels) else f"cluster {c}"))

label_source = ColumnDataSource(dict(zip(("x", "y", "text"), map(list, zip(*label_rows)))))

# --- points -------------------------------------------------------------------
source = ColumnDataSource(dict(
    x=X_2d[:, 0],
    y=X_2d[:, 1],
    color=colors,
    cluster=[str(l) if l >= 0 else "noise" for l in labels],
    filename=df["image_filename"],
    desc=df["beschreibung_en"].fillna("").str.slice(0, 160),
    thumb=df["thumb_html"],
))

TOOLTIP = """
<div style="max-width:260px;">
  @thumb{safe}
  <div><b>cluster @cluster</b></div>
  <div style="color:#555;">@filename</div>
</div>
"""

p = figure(
    width=950, height=750,
    title=f"Photochrom SigLIP2 embeddings — UMAP + HDBSCAN ({n_clusters} clusters)",
    tools="pan,wheel_zoom,box_zoom,reset,save",
    active_scroll="wheel_zoom",
    x_range=(x0, x1), y_range=(y0, y1),
)
p.image_rgba(image=[bg_image], x=x0, y=y0, dw=x1 - x0, dh=y1 - y0)
r = p.scatter("x", "y", source=source, color="color", size=4, alpha=0.75,
              hover_line_color="black", hover_fill_color="color")
p.add_layout(LabelSet(x="x", y="y", text="text", source=label_source,
                      text_color="black", text_font_size="11pt", text_font_style="bold",
                      text_align="center", text_baseline="middle"))
p.add_tools(HoverTool(renderers=[r], tooltips=TOOLTIP))
p.axis.visible = False
p.grid.visible = False
show(p)

Loading BokehJS ...